# MedLoRA · 实验 B: PubMedQA 文本 CPT → SLAKE SFT (Kaggle T4)

两段式: (1) `configs/cpt_pubmed_qlora.yaml` 在 1 万条 PubMedQA 上做 1 epoch CPT; (2) `configs/sft_after_cpt.yaml` 从 CPT adapter 继续做 SLAKE SFT; (3) 三张表评估, 与基线和实验 A 对比。

预计 CPT 约 1 h, SFT 约 4.3 h, 评估约 1.5 h。

In [ ]:
REPO_URL = "https://github.com/AugustLoo/MedLoRA.git"
MODEL = "Qwen/Qwen2.5-VL-3B-Instruct"
CPT = "cpt_pubmed_qlora_r16"
EXP = "sft_after_cpt_r16"
TAG = "cpt_sft_r16"

!git clone -q $REPO_URL /kaggle/working/MedLoRA
%cd /kaggle/working/MedLoRA
!mkdir -p /kaggle/temp/raw && rm -rf data/raw && ln -s /kaggle/temp/raw data/raw
!pip install -q -r requirements.txt
!pip install -q "git+https://github.com/hiyouga/LLaMA-Factory.git"
!llamafactory-cli version
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!python data/download_slake.py | tail -3
!python data/convert_slake_sharegpt.py
!python data/convert_pubmedqa_cpt.py --max 10000
!python -c "import json; d=json.load(open('data/processed/pubmed_cpt.json')); print('cpt docs', len(d)); print(d[0]['text'][:300])"
!cat data/processed/dataset_info.json

In [ ]:
# 阶段 1: CPT (纯文本, 1 epoch)
!CUDA_VISIBLE_DEVICES=0 llamafactory-cli train configs/cpt_pubmed_qlora.yaml 2>&1 | grep -v -E "^\s*$|it/s\]|s/it\]" | tail -40
import glob, shutil
for ck in glob.glob(f"outputs/{CPT}/checkpoint-*"): shutil.rmtree(ck)
!ls outputs/$CPT && du -sh outputs/$CPT

In [ ]:
# 阶段 2: 在 CPT adapter 上继续 SLAKE SFT
!CUDA_VISIBLE_DEVICES=0 llamafactory-cli train configs/sft_after_cpt.yaml 2>&1 | grep -v -E "^\s*$|it/s\]|s/it\]" | tail -60

In [ ]:
import json, glob, os, shutil
for name in [CPT, EXP]:
    log = f"outputs/{name}/trainer_log.jsonl"
    if os.path.exists(log):
        rows = [json.loads(l) for l in open(log)]
        tr = [r for r in rows if "loss" in r]; ev = [r for r in rows if "eval_loss" in r]
        print(name, "train loss first/last:", tr[0]["loss"] if tr else None, tr[-1]["loss"] if tr else None,
              "| eval:", [round(r["eval_loss"], 4) for r in ev], "| NaN:", sum(1 for r in tr if r["loss"] != r["loss"]))
for ck in glob.glob(f"outputs/{EXP}/checkpoint-*"): shutil.rmtree(ck)
!du -sh outputs/$CPT outputs/$EXP

In [ ]:
!CUDA_VISIBLE_DEVICES=0 MODEL=$MODEL bash train/eval_all.sh $TAG outputs/$EXP 2>&1 | grep -v -E "it/s\]|s/it\]"

In [ ]:
import json
base = json.load(open("results/baseline_2026-09-15.json"))
a = json.load(open("results/sft_A_2026-09-15.json"))
b = {k: json.load(open(f"outputs/eval/{k}_{TAG}.json")) for k in ["slake", "textvqa", "pubmedqa"]}
def row(name, x, y, z): print(f"{name:<22}{x:>8}{y:>8}{z:>8}{z-y:>+8.2f}")
print(f"{'metric':<22}{'base':>8}{'A':>8}{'B':>8}{'B-A':>8}")
for k in ["closed_acc", "open_em", "open_recall", "open_f1"]:
    row("slake_" + k, base["slake"]["metrics"][k], a["slake"]["metrics"][k], b["slake"]["metrics"][k])
for m in ["X-Ray", "CT", "MRI"]:
    row(f"slake_closed_{m}", base["slake"]["by_modality"][m]["closed_acc"], a["slake"]["by_modality"][m]["closed_acc"], b["slake"]["by_modality"][m]["closed_acc"])
row("textvqa_acc", base["textvqa"]["textvqa_acc"], a["textvqa"]["textvqa_acc"], b["textvqa"]["textvqa_acc"])
row("pubmedqa_acc", base["pubmedqa"]["accuracy"], a["pubmedqa"]["accuracy"], b["pubmedqa"]["accuracy"])
row("pubmedqa_macro_f1", base["pubmedqa"]["macro_f1"], a["pubmedqa"]["macro_f1"], b["pubmedqa"]["macro_f1"])
print("pubmedqa pred_dist:", b["pubmedqa"]["pred_dist"])
print("RESULTS_JSON", json.dumps(b))